In [1]:
import subprocess
subprocess.run(["pip", "install", "-q", "albumentations==1.4.3"], check=True)

import os, random, warnings, platform, datetime
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, matplotlib.patches as mpatches
import seaborn as sns, cv2

import sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.applications import EfficientNetV2S
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, CSVLogger
from tensorflow.keras.metrics import TopKCategoricalAccuracy

import albumentations as A

warnings.filterwarnings("ignore")

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"
os.environ["TF_CUDNN_DETERMINISTIC"] = "1"
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATASET_DIR = "/kaggle/input/datasets/fisheightcharacter/indonesian-batik-dataset-enhanced-and-cleaned/dataset"
OUTPUT_DIR  = "/kaggle/working"
MODEL_PATH  = os.path.join(OUTPUT_DIR, "best_batik_model_v2.keras")
TFLITE_PATH = os.path.join(OUTPUT_DIR, "batik_model_v2.tflite")
LABELS_PATH = os.path.join(OUTPUT_DIR, "labels.txt")

IMG_HEIGHT, IMG_WIDTH = 224, 224
IMG_SIZE      = (IMG_HEIGHT, IMG_WIDTH)
IMG_CHANNELS  = 3
INPUT_SHAPE   = (IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS)

BATCH_SIZE         = 32
N_FOLDS            = 5
PHASE1_EPOCHS      = 20
PHASE2_EPOCHS      = 40
PHASE1_LR          = 1e-3
PHASE2_LR          = 1e-5
PHASE1_PATIENCE    = 5
PHASE2_PATIENCE    = 7
REDUCE_LR_PATIENCE = 3
REDUCE_LR_FACTOR   = 0.3
MIN_LR             = 1e-7
UNFREEZE_LAYERS    = 40
DROPOUT_RATE_1     = 0.5
DROPOUT_RATE_2     = 0.3
DENSE_UNITS        = 256
LABEL_SMOOTHING    = 0.1
TTA_STEPS          = 5
TEST_SIZE          = 0.15
TIER1_THRESHOLD    = 45
TIER2_THRESHOLD    = 90

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"GPU devices found: {len(gpus)}")
else:
    print("WARNING: No GPU detected. Training will run on CPU.")

tf.keras.mixed_precision.set_global_policy("mixed_float16")
print(f"Mixed precision policy: {tf.keras.mixed_precision.global_policy().name}")

print("\nEnvironment Configuration:")
print(f"  Platform        : {platform.platform()}")
print(f"  Python          : {platform.python_version()}")
print(f"  TensorFlow      : {tf.__version__}")
print(f"  Backbone        : EfficientNetV2S")
print(f"  Dataset path    : {DATASET_DIR}")
print(f"  Image size      : {IMG_SIZE}")
print(f"  Batch size      : {BATCH_SIZE}")
print(f"  K-Folds         : {N_FOLDS}")
print(f"  Label smoothing : {LABEL_SMOOTHING}")
print(f"  TTA steps       : {TTA_STEPS}")
print(f"  Test holdout    : {int(TEST_SIZE * 100)}%")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.0/137.0 kB 2.0 MB/s eta 0:00:00


2026-04-29 13:37:36.466780: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777469856.671002      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777469856.729178      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777469857.210492      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777469857.210530      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777469857.210533      57 computation_placer.cc:177] computation placer alr

GPU devices found: 1
Mixed precision policy: mixed_float16

Environment Configuration:
  Platform        : Linux-6.6.113+-x86_64-with-glibc2.35
  Python          : 3.12.12
  TensorFlow      : 2.19.0
  Backbone        : EfficientNetV2S
  Dataset path    : /kaggle/input/datasets/fisheightcharacter/indonesian-batik-dataset-enhanced-and-cleaned/dataset
  Image size      : (224, 224)
  Batch size      : 32
  K-Folds         : 5
  Label smoothing : 0.1
  TTA steps       : 5
  Test holdout    : 15%


In [2]:
import os, cv2
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

def discover_dataset(root_dir: str) -> pd.DataFrame:
    valid_extensions = {".jpg", ".jpeg", ".png", ".webp"}
    records = []
    root_path = Path(root_dir)

    if not root_path.exists():
        raise FileNotFoundError(f"Dataset directory not found: {root_dir}")

    class_dirs = sorted([d for d in root_path.iterdir() if d.is_dir()])
    if len(class_dirs) == 0:
        raise ValueError(f"No subdirectories found in: {root_dir}")

    for class_dir in class_dirs:
        label = class_dir.name
        for filepath in class_dir.iterdir():
            if filepath.is_file() and filepath.suffix.lower() in valid_extensions:
                records.append({"filepath": str(filepath), "label": label})

    return pd.DataFrame(records)

df = discover_dataset(DATASET_DIR)

label_encoder = LabelEncoder()
df["label_encoded"] = label_encoder.fit_transform(df["label"])
CLASS_NAMES = list(label_encoder.classes_)
NUM_CLASSES = len(CLASS_NAMES)

with open(LABELS_PATH, "w") as f:
    f.write("\n".join(CLASS_NAMES))

print(f"Total images found : {len(df)}")
print(f"Total classes      : {NUM_CLASSES}")

corrupted, dimension_issues = [], []
for idx, row in df.iterrows():
    img = cv2.imread(row["filepath"])
    if img is None:
        corrupted.append(idx)
        continue
    h, w = img.shape[:2]
    if h < 32 or w < 32:
        dimension_issues.append(idx)

if corrupted:
    df.drop(index=corrupted, inplace=True)
if dimension_issues:
    df.drop(index=dimension_issues, inplace=True)

df.reset_index(drop=True, inplace=True)
print(f"Clean dataset size : {len(df)} images")
print(f"Removed corrupted  : {len(corrupted)} | Sub-res: {len(dimension_issues)}")

class_counts = (
    df.groupby("label")
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

class_counts["tier"] = class_counts["count"].apply(
    lambda x: "Tier 1 (Aggressive)" if x <= TIER1_THRESHOLD
    else ("Tier 2 (Moderate)" if x <= TIER2_THRESHOLD else "Tier 3 (Light)")
)

tier_colors = {
    "Tier 1 (Aggressive)": "#d62728",
    "Tier 2 (Moderate)":   "#ff7f0e",
    "Tier 3 (Light)":      "#2ca02c",
}

fig, ax = plt.subplots(figsize=(14, 12))
sorted_counts = class_counts.sort_values("count", ascending=True)
bar_colors = [tier_colors[t] for t in sorted_counts["tier"]]
bars = ax.barh(sorted_counts["label"], sorted_counts["count"], color=bar_colors, edgecolor="white", height=0.75)
for bar, count in zip(bars, sorted_counts["count"]):
    ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height() / 2, str(count), va="center", fontsize=8)
ax.axvline(x=TIER1_THRESHOLD, color="#d62728", linestyle="--", linewidth=1.2)
ax.axvline(x=TIER2_THRESHOLD, color="#ff7f0e", linestyle="--", linewidth=1.2)
ax.set_title("Class Distribution — Batik Dataset v2")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "eda_class_distribution.png"), dpi=150)
plt.close()

SAMPLES_PER_CLASS = 4
fig, axes = plt.subplots(NUM_CLASSES, SAMPLES_PER_CLASS, figsize=(SAMPLES_PER_CLASS * 2.2, NUM_CLASSES * 2.2))
for row_idx, class_name in enumerate(CLASS_NAMES):
    class_df = df[df["label"] == class_name]
    samples = class_df.sample(n=min(SAMPLES_PER_CLASS, len(class_df)), random_state=SEED)
    for col_idx in range(SAMPLES_PER_CLASS):
        ax = axes[row_idx, col_idx]
        if col_idx < len(samples):
            img = cv2.imread(samples.iloc[col_idx]["filepath"])
            if img is not None:
                ax.imshow(cv2.cvtColor(cv2.resize(img, IMG_SIZE), cv2.COLOR_BGR2RGB))
        ax.axis("off")
        if col_idx == 0:
            ax.set_ylabel(class_name, fontsize=7, rotation=0, labelpad=80, va="center")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "eda_sample_grid.png"), dpi=120)
plt.close()

Total images found : 2216
Total classes      : 28
Clean dataset size : 2216 images
Removed corrupted  : 0 | Sub-res: 0


In [4]:
import albumentations as A
import tensorflow as tf
import numpy as np, cv2
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight

# Fixed 15% test holdout — never touched during K-Fold training
X_trainval, X_test, y_trainval, y_test = train_test_split(
    df["filepath"].values,
    df["label_encoded"].values,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=df["label_encoded"].values,
)

print(f"Fixed test holdout : {len(X_test)} images")
print(f"Train+Val pool     : {len(X_trainval)} images")

# K-Fold splitter (applied to trainval pool in Cell 4)
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

# Augmentation pipelines (tier-based)
AUGMENT_TIER1 = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.7),
    A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=40, val_shift_limit=30, p=0.6),
    A.GridDistortion(num_steps=5, distort_limit=0.3, p=0.4),
    A.ElasticTransform(alpha=80, sigma=8, p=0.3),
    A.RandomResizedCrop(size=IMG_SIZE, scale=(0.7, 1.0), ratio=(0.9, 1.1), p=0.5),
    A.GaussianBlur(blur_limit=(3, 5), p=0.2),
    A.CoarseDropout(max_holes=4, min_holes=1, max_height=32, min_height=16, max_width=32, min_width=16, p=0.3),
    A.CLAHE(clip_limit=3.0, tile_grid_size=(8, 8), p=0.3),
    A.Sharpen(alpha=(0.1, 0.3), lightness=(0.8, 1.2), p=0.2),
    A.Resize(height=IMG_HEIGHT, width=IMG_WIDTH),
])

AUGMENT_TIER2 = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.HueSaturationValue(hue_shift_limit=15, sat_shift_limit=25, val_shift_limit=20, p=0.4),
    A.RandomResizedCrop(size=IMG_SIZE, scale=(0.8, 1.0), ratio=(0.9, 1.1), p=0.4),
    A.GaussianBlur(blur_limit=(3, 3), p=0.2),
    A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.2),
    A.Resize(height=IMG_HEIGHT, width=IMG_WIDTH),
])

AUGMENT_TIER3 = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.3),
    A.HueSaturationValue(hue_shift_limit=8, sat_shift_limit=15, val_shift_limit=10, p=0.3),
    A.Resize(height=IMG_HEIGHT, width=IMG_WIDTH),
])

AUGMENT_VALIDATION = A.Compose([A.Resize(height=IMG_HEIGHT, width=IMG_WIDTH)])

# TTA uses light random augmentation (same as Tier 3)
AUGMENT_TTA = AUGMENT_TIER3

def get_tier_for_class(class_name: str) -> str:
    count = df[df["label"] == class_name].shape[0]
    if count <= TIER1_THRESHOLD:
        return "tier1"
    elif count <= TIER2_THRESHOLD:
        return "tier2"
    return "tier3"

CLASS_TIER_MAP = {i: get_tier_for_class(CLASS_NAMES[i]) for i in range(NUM_CLASSES)}

def load_image(filepath: str) -> np.ndarray:
    img = cv2.imread(filepath)
    if img is None:
        img = np.zeros((IMG_HEIGHT, IMG_WIDTH, 3), dtype=np.uint8)
    return cv2.cvtColor(cv2.resize(img, IMG_SIZE), cv2.COLOR_BGR2RGB)

def augment_train(filepath: bytes, label: int) -> tuple:
    img = load_image(filepath.numpy().decode("utf-8"))
    label_int = int(label.numpy())
    tier = CLASS_TIER_MAP.get(label_int, "tier2")
    if tier == "tier1":
        img = AUGMENT_TIER1(image=img)["image"]
    elif tier == "tier2":
        img = AUGMENT_TIER2(image=img)["image"]
    else:
        img = AUGMENT_TIER3(image=img)["image"]
    return img.astype(np.float32), label_int

def augment_eval(filepath: bytes, label: int) -> tuple:
    img = load_image(filepath.numpy().decode("utf-8"))
    img = AUGMENT_VALIDATION(image=img)["image"]
    return img.astype(np.float32), int(label.numpy())

def augment_tta(filepath: bytes, label: int) -> tuple:
    img = load_image(filepath.numpy().decode("utf-8"))
    img = AUGMENT_TTA(image=img)["image"]
    return img.astype(np.float32), int(label.numpy())

def _wrap(aug_fn):
    def tf_fn(filepath, label):
        img, lbl = tf.py_function(func=aug_fn, inp=[filepath, label], Tout=[tf.float32, tf.int32])
        img.set_shape([IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS])
        lbl.set_shape([])
        return preprocess_input(img), tf.one_hot(lbl, depth=NUM_CLASSES)
    return tf_fn

tf_augment_train = _wrap(augment_train)
tf_augment_eval  = _wrap(augment_eval)
tf_augment_tta   = _wrap(augment_tta)

AUTOTUNE = tf.data.AUTOTUNE

def build_dataset(filepaths, labels, augment_fn, batch_size, shuffle=False, cache=False):
    ds = tf.data.Dataset.from_tensor_slices((filepaths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(filepaths), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(augment_fn, num_parallel_calls=AUTOTUNE)
    if cache:
        ds = ds.cache()
    return ds.batch(batch_size).prefetch(AUTOTUNE)

# Build permanent test dataset (used in Cell 6)
test_ds = build_dataset(X_test, y_test, tf_augment_eval, BATCH_SIZE, shuffle=False, cache=True)

print("\nAugmentation tiers:")
for tier, classes in [("Tier 1", [CLASS_NAMES[i] for i, t in CLASS_TIER_MAP.items() if t == "tier1"]),
                       ("Tier 2", [CLASS_NAMES[i] for i, t in CLASS_TIER_MAP.items() if t == "tier2"]),
                       ("Tier 3", [CLASS_NAMES[i] for i, t in CLASS_TIER_MAP.items() if t == "tier3"])]:
    print(f"  {tier} ({len(classes)} classes): {classes}")

Fixed test holdout : 333 images
Train+Val pool     : 1883 images

Augmentation tiers:
  Tier 1 (1 classes): ['Priangan_Merak_Ngibing']
  Tier 2 (22 classes): ['Bali_Barong', 'Bali_Merak', 'Ceplok', 'Corak_Insang', 'Ikat_Celup', 'Jakarta_Ondel_Ondel', 'Jawa_Timur_Pring', 'Lampung_Gajah', 'Lasem', 'Madura_Mataketeran', 'Maluku_Pala', 'NTB_Lumbung', 'Papua_Asmat', 'Papua_Tifa', 'Sekar', 'Sidoluhur', 'Sogan', 'Sulawesi_Selatan_Lontara', 'Sumatera_Barat_Rumah_Minang', 'Sumatera_Utara_Boraspati', 'Tambal', 'Yogyakarta_Parang']
  Tier 3 (5 classes): ['Jawa_Barat_Megamendung', 'Kalimantan_Dayak', 'Papua_Cendrawasih', 'Solo_Parang', 'Yogyakarta_Kawung']


In [5]:
import tensorflow as tf
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import os, gc

def build_model(num_classes: int) -> tuple:
    inputs = tf.keras.Input(shape=INPUT_SHAPE, name="input_image")
    base = EfficientNetV2S(weights="imagenet", include_top=False, input_tensor=inputs)
    base.trainable = False
    x = base.output
    x = GlobalAveragePooling2D(name="global_avg_pool")(x)
    x = BatchNormalization(name="head_bn")(x)
    x = Dropout(DROPOUT_RATE_1, name="dropout_1")(x)
    x = Dense(DENSE_UNITS, activation="relu", name="dense_head")(x)
    x = Dropout(DROPOUT_RATE_2, name="dropout_2")(x)
    x = Dense(num_classes, name="dense_logits")(x)
    outputs = tf.keras.layers.Activation("softmax", dtype="float32", name="predictions")(x)
    model = Model(inputs=inputs, outputs=outputs, name="batik_classifier_v2")
    return base, model

fold_results = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_trainval, y_trainval)):
    print(f"\n{'='*60}")
    print(f"  FOLD {fold + 1} / {N_FOLDS}")
    print(f"{'='*60}")

    X_tr, X_val = X_trainval[train_idx], X_trainval[val_idx]
    y_tr, y_val = y_trainval[train_idx], y_trainval[val_idx]

    class_weights_arr = compute_class_weight(class_weight="balanced", classes=np.arange(NUM_CLASSES), y=y_tr)
    fold_class_weights = dict(enumerate(class_weights_arr))

    train_ds_fold = build_dataset(X_tr, y_tr, tf_augment_train, BATCH_SIZE, shuffle=True)
    val_ds_fold   = build_dataset(X_val, y_val, tf_augment_eval, BATCH_SIZE, shuffle=False)

    base_model, model = build_model(NUM_CLASSES)

    # Phase 1: frozen backbone, train head only
    model.compile(
        optimizer=Adam(learning_rate=PHASE1_LR),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
        metrics=["accuracy", TopKCategoricalAccuracy(k=3, name="top3_accuracy")],
    )

    print(f"  Phase 1: Training head...")
    history1 = model.fit(
        train_ds_fold,
        validation_data=val_ds_fold,
        epochs=PHASE1_EPOCHS,
        callbacks=[
            EarlyStopping(monitor="val_accuracy", patience=PHASE1_PATIENCE, restore_best_weights=True, verbose=0, mode="max"),
            ReduceLROnPlateau(monitor="val_loss", factor=REDUCE_LR_FACTOR, patience=REDUCE_LR_PATIENCE, min_lr=MIN_LR, verbose=0),
        ],
        class_weight=fold_class_weights,
        verbose=1,
    )

    # Phase 2: unfreeze last N layers, keep BatchNorm frozen
    base_model.trainable = True
    for layer in base_model.layers[:-UNFREEZE_LAYERS]:
        layer.trainable = False
    for layer in base_model.layers:
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False

    fold_model_path = os.path.join(OUTPUT_DIR, f"fold_{fold + 1}_best.keras")

    model.compile(
        optimizer=Adam(learning_rate=PHASE2_LR),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
        metrics=["accuracy", TopKCategoricalAccuracy(k=3, name="top3_accuracy")],
    )

    print(f"  Phase 2: Fine-tuning last {UNFREEZE_LAYERS} layers...")
    history2 = model.fit(
        train_ds_fold,
        validation_data=val_ds_fold,
        epochs=PHASE2_EPOCHS,
        callbacks=[
            ModelCheckpoint(filepath=fold_model_path, monitor="val_accuracy", save_best_only=True, mode="max", verbose=0),
            EarlyStopping(monitor="val_accuracy", patience=PHASE2_PATIENCE, restore_best_weights=True, verbose=0, mode="max"),
            ReduceLROnPlateau(monitor="val_loss", factor=REDUCE_LR_FACTOR, patience=REDUCE_LR_PATIENCE, min_lr=MIN_LR, verbose=0),
        ],
        class_weight=fold_class_weights,
        verbose=1,
    )

    best_val_acc  = max(history2.history["val_accuracy"])
    best_val_top3 = max(history2.history["val_top3_accuracy"])

    fold_results.append({"fold": fold + 1, "val_accuracy": best_val_acc, "val_top3": best_val_top3})
    print(f"\n  Fold {fold + 1} -> val_acc: {best_val_acc:.4f} | val_top3: {best_val_top3:.4f}")

    del model, base_model, train_ds_fold, val_ds_fold
    tf.keras.backend.clear_session()
    gc.collect()


  FOLD 1 / 5
82420632/82420632 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step
  Phase 1: Training head...
Epoch 1/20


E0000 00:00:1777469983.154435      57 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/batik_classifier_v2_1/block1b_drop_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer
I0000 00:00:1777469985.158095     145 cuda_dnn.cc:529] Loaded cuDNN version 91002


48/48 ━━━━━━━━━━━━━━━━━━━━ 64s 730ms/step - accuracy: 0.1420 - loss: 4.1068 - top3_accuracy: 0.2766 - val_accuracy: 0.5889 - val_loss: 2.3389 - val_top3_accuracy: 0.7984 - learning_rate: 0.0010
Epoch 2/20
48/48 ━━━━━━━━━━━━━━━━━━━━ 23s 479ms/step - accuracy: 0.4576 - loss: 2.4241 - top3_accuracy: 0.6775 - val_accuracy: 0.6790 - val_loss: 1.9933 - val_top3_accuracy: 0.8408 - learning_rate: 0.0010
Epoch 3/20
48/48 ━━━━━━━━━━━━━━━━━━━━ 23s 477ms/step - accuracy: 0.5602 - loss: 2.0792 - top3_accuracy: 0.7777 - val_accuracy: 0.7109 - val_loss: 1.7319 - val_top3_accuracy: 0.8806 - learning_rate: 0.0010
Epoch 4/20
48/48 ━━━━━━━━━━━━━━━━━━━━ 23s 479ms/step - accuracy: 0.5860 - loss: 1.9729 - top3_accuracy: 0.8216 - val_accuracy: 0.7294 - val_loss: 1.5545 - val_top3_accuracy: 0.8939 - learning_rate: 0.0010
Epoch 5/20
48/48 ━━━━━━━━━━━━━━━━━━━━ 23s 478ms/step - accuracy: 0.6743 - loss: 1.7735 - top3_accuracy: 0.8545 - val_accuracy: 0.7401 - val_loss: 1.4799 - val_top3_accuracy: 0.8992 - learning

E0000 00:00:1777470418.503588      57 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/batik_classifier_v2_1/block1b_drop_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


48/48 ━━━━━━━━━━━━━━━━━━━━ 58s 691ms/step - accuracy: 0.7860 - loss: 1.3635 - top3_accuracy: 0.9273 - val_accuracy: 0.8488 - val_loss: 1.3033 - val_top3_accuracy: 0.9310 - learning_rate: 1.0000e-05
Epoch 2/40
48/48 ━━━━━━━━━━━━━━━━━━━━ 25s 524ms/step - accuracy: 0.7836 - loss: 1.3569 - top3_accuracy: 0.9424 - val_accuracy: 0.8568 - val_loss: 1.2945 - val_top3_accuracy: 0.9310 - learning_rate: 1.0000e-05
Epoch 3/40
48/48 ━━━━━━━━━━━━━━━━━━━━ 26s 534ms/step - accuracy: 0.7795 - loss: 1.3428 - top3_accuracy: 0.9426 - val_accuracy: 0.8594 - val_loss: 1.2923 - val_top3_accuracy: 0.9310 - learning_rate: 1.0000e-05
Epoch 4/40
48/48 ━━━━━━━━━━━━━━━━━━━━ 25s 531ms/step - accuracy: 0.7801 - loss: 1.3302 - top3_accuracy: 0.9260 - val_accuracy: 0.8647 - val_loss: 1.2907 - val_top3_accuracy: 0.9310 - learning_rate: 1.0000e-05
Epoch 5/40
48/48 ━━━━━━━━━━━━━━━━━━━━ 23s 488ms/step - accuracy: 0.7941 - loss: 1.3310 - top3_accuracy: 0.9382 - val_accuracy: 0.8647 - val_loss: 1.2851 - val_top3_accuracy: 0

...

48/48 ━━━━━━━━━━━━━━━━━━━━ 55s 676ms/step - accuracy: 0.8121 - loss: 1.2290 - top3_accuracy: 0.9455 - val_accuracy: 0.8059 - val_loss: 1.3419 - val_top3_accuracy: 0.9122 - learning_rate: 1.0000e-05
Epoch 2/40
48/48 ━━━━━━━━━━━━━━━━━━━━ 22s 472ms/step - accuracy: 0.8319 - loss: 1.2507 - top3_accuracy: 0.9519 - val_accuracy: 0.8059 - val_loss: 1.3349 - val_top3_accuracy: 0.9149 - learning_rate: 1.0000e-05
Epoch 3/40
48/48 ━━━━━━━━━━━━━━━━━━━━ 24s 507ms/step - accuracy: 0.8157 - loss: 1.2192 - top3_accuracy: 0.9534 - val_accuracy: 0.8112 - val_loss: 1.3330 - val_top3_accuracy: 0.9202 - learning_rate: 1.0000e-05
Epoch 4/40
48/48 ━━━━━━━━━━━━━━━━━━━━ 23s 482ms/step - accuracy: 0.8211 - loss: 1.2329 - top3_accuracy: 0.9521 - val_accuracy: 0.8112 - val_loss: 1.3324 - val_top3_accuracy: 0.9176 - learning_rate: 1.0000e-05
Epoch 5/40
48/48 ━━━━━━━━━━━━━━━━━━━━ 23s 468ms/step - accuracy: 0.8093 - loss: 1.2844 - top3_accuracy: 0.9553 - val_accuracy: 0.8112 - val_loss: 1.3287 - val_top3_accuracy: 0

E0000 00:00:1777473639.559041      57 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/batik_classifier_v2_1/block1b_drop_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


48/48 ━━━━━━━━━━━━━━━━━━━━ 47s 581ms/step - accuracy: 0.1571 - loss: 4.1579 - top3_accuracy: 0.2981 - val_accuracy: 0.5133 - val_loss: 2.3953 - val_top3_accuracy: 0.7261 - learning_rate: 0.0010
Epoch 2/20
48/48 ━━━━━━━━━━━━━━━━━━━━ 22s 463ms/step - accuracy: 0.4814 - loss: 2.4247 - top3_accuracy: 0.6982 - val_accuracy: 0.6463 - val_loss: 2.0487 - val_top3_accuracy: 0.7899 - learning_rate: 0.0010
Epoch 3/20
48/48 ━━━━━━━━━━━━━━━━━━━━ 22s 460ms/step - accuracy: 0.5766 - loss: 2.0981 - top3_accuracy: 0.7740 - val_accuracy: 0.6676 - val_loss: 1.8370 - val_top3_accuracy: 0.8271 - learning_rate: 0.0010
Epoch 4/20
48/48 ━━━━━━━━━━━━━━━━━━━━ 22s 453ms/step - accuracy: 0.6291 - loss: 1.8665 - top3_accuracy: 0.8278 - val_accuracy: 0.6941 - val_loss: 1.7076 - val_top3_accuracy: 0.8271 - learning_rate: 0.0010
Epoch 5/20
48/48 ━━━━━━━━━━━━━━━━━━━━ 22s 461ms/step - accuracy: 0.6657 - loss: 1.7234 - top3_accuracy: 0.8463 - val_accuracy: 0.7021 - val_loss: 1.6208 - val_top3_accuracy: 0.8511 - learning

E0000 00:00:1777474103.660058      57 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/batik_classifier_v2_1/block1b_drop_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


48/48 ━━━━━━━━━━━━━━━━━━━━ 50s 612ms/step - accuracy: 0.8195 - loss: 1.2715 - top3_accuracy: 0.9483 - val_accuracy: 0.8112 - val_loss: 1.4566 - val_top3_accuracy: 0.8963 - learning_rate: 1.0000e-05
Epoch 2/40
48/48 ━━━━━━━━━━━━━━━━━━━━ 22s 457ms/step - accuracy: 0.8107 - loss: 1.2799 - top3_accuracy: 0.9553 - val_accuracy: 0.8085 - val_loss: 1.4510 - val_top3_accuracy: 0.8989 - learning_rate: 1.0000e-05
Epoch 3/40
48/48 ━━━━━━━━━━━━━━━━━━━━ 22s 460ms/step - accuracy: 0.8344 - loss: 1.2260 - top3_accuracy: 0.9592 - val_accuracy: 0.8085 - val_loss: 1.4472 - val_top3_accuracy: 0.8989 - learning_rate: 1.0000e-05
Epoch 4/40
48/48 ━━━━━━━━━━━━━━━━━━━━ 24s 494ms/step - accuracy: 0.8502 - loss: 1.2393 - top3_accuracy: 0.9626 - val_accuracy: 0.8165 - val_loss: 1.4440 - val_top3_accuracy: 0.8989 - learning_rate: 1.0000e-05
Epoch 5/40
48/48 ━━━━━━━━━━━━━━━━━━━━ 22s 453ms/step - accuracy: 0.8361 - loss: 1.1994 - top3_accuracy: 0.9522 - val_accuracy: 0.8112 - val_loss: 1.4380 - val_top3_accuracy: 0

In [6]:
import tensorflow as tf
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import os, gc

# K-Fold Cross-Validation Summary
results_df = pd.DataFrame(fold_results)

print("\n" + "=" * 55)
print("  K-FOLD CROSS-VALIDATION SUMMARY")
print("=" * 55)
print(results_df.to_string(index=False))
print(f"\n  Mean Val Accuracy : {results_df['val_accuracy'].mean():.4f} ± {results_df['val_accuracy'].std():.4f}")
print(f"  Mean Val Top-3    : {results_df['val_top3'].mean():.4f} ± {results_df['val_top3'].std():.4f}")
print("=" * 55)

# Final model: train on 100% of trainval data using best hyperparams proven by K-Fold
print("\nFinal Model: Training on 100% of trainval data...")

class_weights_arr = compute_class_weight(class_weight="balanced", classes=np.arange(NUM_CLASSES), y=y_trainval)
final_class_weights = dict(enumerate(class_weights_arr))

full_train_ds = build_dataset(X_trainval, y_trainval, tf_augment_train, BATCH_SIZE, shuffle=True)

base_model, model = build_model(NUM_CLASSES)

# Phase 1: head only — monitored on training accuracy since no validation set
model.compile(
    optimizer=Adam(learning_rate=PHASE1_LR),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
    metrics=["accuracy", TopKCategoricalAccuracy(k=3, name="top3_accuracy")],
)

print("  Phase 1: Training head...")
model.fit(
    full_train_ds,
    epochs=PHASE1_EPOCHS,
    callbacks=[
        EarlyStopping(monitor="accuracy", patience=PHASE1_PATIENCE, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="loss", factor=REDUCE_LR_FACTOR, patience=REDUCE_LR_PATIENCE, min_lr=MIN_LR, verbose=1),
        CSVLogger(os.path.join(OUTPUT_DIR, "final_phase1_log.csv"), append=False),
    ],
    class_weight=final_class_weights,
    verbose=1,
)

# Phase 2: fine-tune
base_model.trainable = True
for layer in base_model.layers[:-UNFREEZE_LAYERS]:
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(
    optimizer=Adam(learning_rate=PHASE2_LR),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
    metrics=["accuracy", TopKCategoricalAccuracy(k=3, name="top3_accuracy")],
)

print("  Phase 2: Fine-tuning...")
model.fit(
    full_train_ds,
    epochs=PHASE2_EPOCHS,
    callbacks=[
        ModelCheckpoint(filepath=MODEL_PATH, monitor="accuracy", save_best_only=True, mode="max", verbose=1),
        EarlyStopping(monitor="accuracy", patience=PHASE2_PATIENCE, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="loss", factor=REDUCE_LR_FACTOR, patience=REDUCE_LR_PATIENCE, min_lr=MIN_LR, verbose=1),
        CSVLogger(os.path.join(OUTPUT_DIR, "final_phase2_log.csv"), append=False),
    ],
    class_weight=final_class_weights,
    verbose=1,
)

print(f"\nFinal model saved to: {MODEL_PATH}")


  K-FOLD CROSS-VALIDATION SUMMARY
 fold  val_accuracy  val_top3
    1      0.867374  0.933687
    2      0.838196  0.931035
    3      0.811671  0.920424
    4      0.819149  0.925532
    5      0.816489  0.898936

  Mean Val Accuracy : 0.8306 ± 0.0229
  Mean Val Top-3    : 0.9219 ± 0.0138

Final Model: Training on 100% of trainval data...
  Phase 1: Training head...
Epoch 1/20


E0000 00:00:1777474418.530080      57 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/batik_classifier_v2_1/block1b_drop_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


59/59 ━━━━━━━━━━━━━━━━━━━━ 43s 385ms/step - accuracy: 0.1776 - loss: 4.0142 - top3_accuracy: 0.3276 - learning_rate: 0.0010
Epoch 2/20
59/59 ━━━━━━━━━━━━━━━━━━━━ 23s 382ms/step - accuracy: 0.5022 - loss: 2.3020 - top3_accuracy: 0.7270 - learning_rate: 0.0010
Epoch 3/20
59/59 ━━━━━━━━━━━━━━━━━━━━ 23s 395ms/step - accuracy: 0.5875 - loss: 1.9825 - top3_accuracy: 0.8135 - learning_rate: 0.0010
Epoch 4/20
59/59 ━━━━━━━━━━━━━━━━━━━━ 23s 396ms/step - accuracy: 0.6167 - loss: 1.9423 - top3_accuracy: 0.8260 - learning_rate: 0.0010
Epoch 5/20
59/59 ━━━━━━━━━━━━━━━━━━━━ 23s 381ms/step - accuracy: 0.6616 - loss: 1.7639 - top3_accuracy: 0.8586 - learning_rate: 0.0010
Epoch 6/20
59/59 ━━━━━━━━━━━━━━━━━━━━ 23s 383ms/step - accuracy: 0.7000 - loss: 1.6665 - top3_accuracy: 0.8626 - learning_rate: 0.0010
Epoch 7/20
59/59 ━━━━━━━━━━━━━━━━━━━━ 23s 384ms/step - accuracy: 0.7036 - loss: 1.5944 - top3_accuracy: 0.8960 - learning_rate: 0.0010
Epoch 8/20
59/59 ━━━━━━━━━━━━━━━━━━━━ 22s 381ms/step - accuracy: 0

E0000 00:00:1777474886.826176      57 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/batik_classifier_v2_1/block1b_drop_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


59/59 ━━━━━━━━━━━━━━━━━━━━ 0s 375ms/step - accuracy: 0.8485 - loss: 1.2108 - top3_accuracy: 0.9589
Epoch 1: accuracy improved from -inf to 0.84546, saving model to /kaggle/working/best_batik_model_v2.keras
59/59 ━━━━━━━━━━━━━━━━━━━━ 45s 407ms/step - accuracy: 0.8484 - loss: 1.2108 - top3_accuracy: 0.9590 - learning_rate: 1.0000e-05
Epoch 2/40
59/59 ━━━━━━━━━━━━━━━━━━━━ 0s 376ms/step - accuracy: 0.8411 - loss: 1.2191 - top3_accuracy: 0.9625
Epoch 2: accuracy did not improve from 0.84546
59/59 ━━━━━━━━━━━━━━━━━━━━ 22s 377ms/step - accuracy: 0.8411 - loss: 1.2190 - top3_accuracy: 0.9625 - learning_rate: 1.0000e-05
Epoch 3/40
59/59 ━━━━━━━━━━━━━━━━━━━━ 0s 374ms/step - accuracy: 0.8668 - loss: 1.1764 - top3_accuracy: 0.9644
Epoch 3: accuracy improved from 0.84546 to 0.86086, saving model to /kaggle/working/best_batik_model_v2.keras
59/59 ━━━━━━━━━━━━━━━━━━━━ 24s 404ms/step - accuracy: 0.8667 - loss: 1.1768 - top3_accuracy: 0.9644 - learning_rate: 1.0000e-05
Epoch 4/40
59/59 ━━━━━━━━━━━━━━━━

In [7]:
import os
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import tensorflow as tf, cv2, gc
from sklearn.metrics import classification_report, confusion_matrix

print("Loading final model for evaluation...")
best_model = tf.keras.models.load_model(MODEL_PATH)

# Standard evaluation (no augmentation)
test_loss, test_acc, test_top3 = best_model.evaluate(test_ds, verbose=1)
print(f"\nStandard Eval -> Loss: {test_loss:.4f} | Acc: {test_acc:.4f} | Top-3: {test_top3:.4f}")

# Test-Time Augmentation (TTA_STEPS passes with light random augmentation)
print(f"\nRunning TTA ({TTA_STEPS} passes)...")

tta_preds = np.zeros((len(X_test), NUM_CLASSES))

for step in range(TTA_STEPS):
    tta_ds = build_dataset(X_test, y_test, tf_augment_tta, BATCH_SIZE, shuffle=False)
    tta_preds += best_model.predict(tta_ds, verbose=0)
    print(f"  TTA pass {step + 1}/{TTA_STEPS} done")

tta_preds /= TTA_STEPS
y_pred = np.argmax(tta_preds, axis=1)
y_true = y_test

tta_acc = np.mean(y_pred == y_true)
print(f"\nStandard accuracy : {test_acc:.4f}")
print(f"TTA accuracy      : {tta_acc:.4f}  (delta: {tta_acc - test_acc:+.4f})")

report_str = classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4)
print("\nClassification Report:\n", report_str)

with open(os.path.join(OUTPUT_DIR, "test_classification_report.txt"), "w") as f:
    f.write(f"Standard Accuracy : {test_acc:.4f}\n")
    f.write(f"TTA Accuracy      : {tta_acc:.4f}\n\n")
    f.write(report_str)

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(24, 20))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title("Confusion Matrix — Test Set (with TTA)", fontsize=18)
plt.xticks(rotation=90)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "eval_confusion_matrix.png"), dpi=150)
plt.close()

# Per-class F1 bar chart
report_dict = classification_report(y_true, y_pred, target_names=CLASS_NAMES, output_dict=True)
f1_df = pd.DataFrame(
    [{"Class": cls, "F1_Score": report_dict[cls]["f1-score"]} for cls in CLASS_NAMES]
).sort_values(by="F1_Score")

colors = ["#d62728" if x < 0.70 else "#ff7f0e" if x < 0.85 else "#2ca02c" for x in f1_df["F1_Score"]]
plt.figure(figsize=(16, 10))
plt.barh(f1_df["Class"], f1_df["F1_Score"], color=colors, edgecolor="white")
plt.axvline(x=tta_acc, color="black", linestyle="--", label=f"TTA Accuracy ({tta_acc:.3f})")
plt.axvline(x=0.85, color="green", linestyle=":", alpha=0.6)
plt.axvline(x=0.70, color="red",   linestyle=":", alpha=0.6)
plt.title("Per-Class F1 Score (TTA Evaluation)")
plt.xlim(0, 1.05)
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "eval_per_class_f1.png"), dpi=120)
plt.close()

print("\nEvaluation Complete.")

Loading final model for evaluation...


2026-04-29 15:05:56.708116: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_4}}


11/11 ━━━━━━━━━━━━━━━━━━━━ 12s 344ms/step - accuracy: 0.8301 - loss: 1.2106 - top3_accuracy: 0.9382

Standard Eval -> Loss: 1.2230 | Acc: 0.8378 | Top-3: 0.9399

Running TTA (5 passes)...
  TTA pass 1/5 done
  TTA pass 2/5 done
  TTA pass 3/5 done
  TTA pass 4/5 done
  TTA pass 5/5 done

Standard accuracy : 0.8378
TTA accuracy      : 0.8468  (delta: +0.0090)

Classification Report:
                              precision    recall  f1-score   support

                Bali_Barong     1.0000    1.0000    1.0000         8
                 Bali_Merak     0.8000    0.8889    0.8421         9
                     Ceplok     0.8333    0.5556    0.6667         9
               Corak_Insang     0.8571    1.0000    0.9231        12
                 Ikat_Celup     0.8750    0.7778    0.8235         9
        Jakarta_Ondel_Ondel     1.0000    0.8750    0.9333         8
     Jawa_Barat_Megamendung     1.0000    1.0000    1.0000        32
           Jawa_Timur_Pring     0.6000    0.7500    0.6667   

In [8]:
import os
import tensorflow as tf
import gc

print("\nRebuilding model in float32 to bypass mixed-precision TFLite trace deadlock...")
tf.keras.mixed_precision.set_global_policy("float32")

# Rebuild EfficientNetV2S architecture in pure float32 to remove dynamic Cast ops
inputs = tf.keras.Input(shape=INPUT_SHAPE, name="input_image")
base_model_fp32 = tf.keras.applications.EfficientNetV2S(weights=None, include_top=False, input_tensor=inputs)
x = base_model_fp32.output
x = tf.keras.layers.GlobalAveragePooling2D(name="global_avg_pool")(x)
x = tf.keras.layers.BatchNormalization(name="head_bn")(x)
x = tf.keras.layers.Dropout(DROPOUT_RATE_1, name="dropout_1")(x)
x = tf.keras.layers.Dense(DENSE_UNITS, activation="relu", name="dense_head")(x)
x = tf.keras.layers.Dropout(DROPOUT_RATE_2, name="dropout_2")(x)
x = tf.keras.layers.Dense(NUM_CLASSES, name="dense_logits")(x)
outputs = tf.keras.layers.Activation("softmax", dtype="float32", name="predictions")(x)
model_fp32 = tf.keras.models.Model(inputs=inputs, outputs=outputs)

print("Loading trained weights into float32 graph...")
model_fp32.load_weights(MODEL_PATH)

print("Freeing GPU memory to prevent TFLiteConverter hang...")
if "best_model" in dir():
    del best_model
tf.keras.backend.clear_session()
gc.collect()

print("Converting to TFLite (FP16) on CPU directly from Keras model...")
with tf.device("/CPU:0"):
    converter = tf.lite.TFLiteConverter.from_keras_model(model_fp32)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.target_spec.supported_types = [tf.float16]
    tflite_model = converter.convert()

with open(TFLITE_PATH, "wb") as f:
    f.write(tflite_model)

keras_size  = os.path.getsize(MODEL_PATH) / (1024 * 1024)
tflite_size = os.path.getsize(TFLITE_PATH) / (1024 * 1024)

print(f"\nKeras model size : {keras_size:.1f} MB")
print(f"TFLite size      : {tflite_size:.1f} MB")
print("V2 Pipeline Complete. TFLite model is ready for deployment.")



Rebuilding model in float32 to bypass mixed-precision TFLite trace deadlock...
Loading trained weights into float32 graph...
Freeing GPU memory to prevent TFLiteConverter hang...
Converting to TFLite (FP16) on CPU directly from Keras model...
INFO:tensorflow:Assets written to: /tmp/tmp756lem8d/assets


INFO:tensorflow:Assets written to: /tmp/tmp756lem8d/assets


Saved artifact at '/tmp/tmp756lem8d'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_image')
Output Type:
  TensorSpec(shape=(None, 28), dtype=tf.float32, name=None)
Captures:
  132584449541520: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132580464384272: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132580464389456: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132584449549200: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132584449550544: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132580464384464: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132580464381200: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132580464385040: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132580464380624: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132580464390608: TensorSpec(shape=(), dtype=tf.resource, name=None)
  13258046438081

W0000 00:00:1777475309.067664      57 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1777475309.067720      57 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
I0000 00:00:1777475309.524030      57 mlir_graph_optimization_pass.cc:425] MLIR V1 optimization pass is not enabled



Keras model size : 105.4 MB
TFLite size      : 39.2 MB
V2 Pipeline Complete. TFLite model is ready for deployment.


In [9]:
import os, zipfile, shutil

OUTPUT_DIR = "/kaggle/working"
ZIP_PATH   = os.path.join(OUTPUT_DIR, "batik_v2_outputs.zip")

# Extensions worth keeping
KEEP_EXTS = {".keras", ".tflite", ".txt", ".csv", ".png", ".jpg"}

print(f"Zipping output files to: {ZIP_PATH}\n")

with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for fname in sorted(os.listdir(OUTPUT_DIR)):
        fpath = os.path.join(OUTPUT_DIR, fname)
        if not os.path.isfile(fpath):
            continue
        if os.path.splitext(fname)[1].lower() not in KEEP_EXTS:
            continue
        if fname == os.path.basename(ZIP_PATH):
            continue
        zf.write(fpath, arcname=fname)
        size_mb = os.path.getsize(fpath) / (1024 * 1024)
        print(f"  Added: {fname:55s} ({size_mb:.2f} MB)")

zip_size_mb = os.path.getsize(ZIP_PATH) / (1024 * 1024)
print(f"\nZip created: {ZIP_PATH}  ({zip_size_mb:.2f} MB)")
print("Download it from the Kaggle Output panel on the right.")


Zipping output files to: /kaggle/working/batik_v2_outputs.zip

  Added: batik_model_v2.tflite                                   (39.17 MB)
  Added: best_batik_model_v2.keras                               (105.41 MB)
  Added: eda_class_distribution.png                              (0.15 MB)
  Added: eda_sample_grid.png                                     (14.88 MB)
  Added: eval_confusion_matrix.png                               (0.31 MB)
  Added: eval_per_class_f1.png                                   (0.11 MB)
  Added: final_phase1_log.csv                                    (0.00 MB)
  Added: final_phase2_log.csv                                    (0.00 MB)
  Added: fold_1_best.keras                                       (130.26 MB)
  Added: fold_2_best.keras                                       (105.41 MB)
  Added: fold_3_best.keras                                       (105.41 MB)
  Added: fold_4_best.keras                                       (105.41 MB)
  Added: fold_5_best.kera